# Appendix F: GMS Training and Post-Training Diagnostics

**Monograph:** *How to Test and Validate Agentic AI Systems*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Prepare the four training data streams from a knowledge graph (positives, hard negatives, tension pairs, relation triangles).
2. Configure the multi-component loss with validated hyperparameters.
3. Train a GMS store and monitor loss convergence.
4. Run the nine-check post-training diagnostic suite.
5. Interpret diagnostic failures and apply targeted fixes.

## Prerequisites

- Chapter 2 notebook completed (familiarity with GMS primitives and the running-example store).
- No API keys required. All exercises use the bundled `model_risk_assessment.md`.

---
## Preamble

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.testing import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


---
## F.1 Ingest the Running-Example Document

We start by building a GMS store from the model-risk-assessment document used throughout this monograph.

In [ ]:
from importlib.resources import files
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="appF_training",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()

stats = store.stats()
print(f"Entities:  {stats['entities']}")
print(f"Relations: {stats['relations']}")
print(f"Triples:   {stats['triples']}")
print(f"ENM:       {stats['enm_entries']}")

---
## F.2 The Multi-Component Loss

The full GMS training loss combines five components:

$$L = \lambda_{\text{geo}} L_{\text{geo}} + \lambda_{\text{tension}} L_{\text{tension}} + \lambda_{\text{path}} L_{\text{path}} + \lambda_{\text{logic}} L_{\text{logic}} + \lambda_{\text{recon}} L_{\text{recon}}$$

| Component | Space | Primitives |
|-----------|-------|------------|
| $L_{\text{geo}}$ | $v$ (semantic) | `score_triple`, `link_predict` |
| $L_{\text{tension}}$ | $u$ (logical) | `tension_energy` |
| $L_{\text{path}}$ | $v$ (semantic) | `check_holonomy` |
| $L_{\text{logic}}$ | rotors | relational composition |
| $L_{\text{recon}}$ | $v$ (semantic) | compression memory |

**Key insight:** $v$-space and $u$-space are independent. Without explicit tension supervision ($\lambda_{\text{tension}} = 0$), the logical embeddings remain random and `tension_energy` returns values clustered near $\sqrt{2}$.

In [ ]:
from knowlytix.core.config import LossConfig

cfg = LossConfig()
print("=== Validated Loss Defaults ===")
for field_name in ['lambda_geo', 'lambda_tension', 'lambda_neg',
                    'gamma', 'adv_temperature', 'lambda_path']:
    print(f"  {field_name:20s} = {getattr(cfg, field_name)}")

---
## F.3 Data Preparation

Training requires four data streams mined automatically from the knowledge graph:
1. **Positive triples** -- every $(h, r, t)$ drives geodesic distance toward zero.
2. **Negative triples** -- type-aware hard sampling with self-negative, hard and random pools.
3. **Tension pairs** -- agreement, contradiction and irrelevance labels for $u$-space.
4. **Relation triangles** -- for holonomy composition training.

In [ ]:
from knowlytix.core.data.prepare import prepare_training_data, sample_tension_batch

adapter = store.adapter
training_data = prepare_training_data(adapter, num_neg=32)

print(f"Positive triples:      {len(training_data.dataset)}")
print(f"Agreement pairs:       {len(training_data.agree_pairs)}")
print(f"Contradiction pairs:   {len(training_data.contra_pairs)}")
print(f"Relation triangles:    {len(training_data.relation_triangles)}")
print(f"Entities:              {training_data.n_entities}")

### F.3.1 Inspecting Negative Sampling

Each positive triple generates $k$ negatives split into three pools: 1 self-negative, $\lfloor(k-1)/2\rfloor$ type-aware hard negatives and the rest uniform random.

In [6]:
# Inspect a single training sample
h_idx, r_idx, t_idx, neg_tails = training_data.dataset[0]

idx_to_ent = {v: k for k, v in adapter.entity_to_idx.items()}
idx_to_rel = {v: k for k, v in adapter.relation_to_idx.items()}

print(f"Positive: ({idx_to_ent[h_idx.item()]}, {idx_to_rel[r_idx.item()]}, {idx_to_ent[t_idx.item()]})")
print(f"\nNegative tails ({len(neg_tails)} total):")
print(f"  Self-negative (slot 0): {idx_to_ent.get(neg_tails[0].item(), neg_tails[0].item())}")
print(f"  Hard negatives (slots 1-{len(neg_tails)//2}):")
for j in range(1, min(4, len(neg_tails)//2 + 1)):
    print(f"    {idx_to_ent.get(neg_tails[j].item(), neg_tails[j].item())}")
print(f"  Random negatives (slots {len(neg_tails)//2+1}-{len(neg_tails)-1}):")
for j in range(len(neg_tails)//2 + 1, min(len(neg_tails)//2 + 4, len(neg_tails))):
    print(f"    {idx_to_ent.get(neg_tails[j].item(), neg_tails[j].item())}")


Positive: (Organization, has_detail, Meridian Financial Technologies, Inc.)

Negative tails (32 total):
  Self-negative (slot 0): Organization
  Hard negatives (slots 1-16):
    5
    Model Validation Unit (MVU), 8 senior validators
    47
  Random negatives (slots 17-31):
    0.8034
    MDL-001/Gender: Female/74.56%/77.23%
    +0.66%


### F.3.2 Inspecting Tension Pairs

Tension pairs are mined from the KG structure. Agreement pairs share $(r, t)$; contradiction pairs have different tails for the same relation.

In [7]:
# Show example tension pairs
print("=== Agreement Pairs (target E* = 0) ===")
for a, b in training_data.agree_pairs[:5]:
    print(f"  {a}  <-->  {b}")

print("\n=== Contradiction Pairs (target E* = 2) ===")
for a, b in training_data.contra_pairs[:5]:
    print(f"  {a}  <-->  {b}")

=== Agreement Pairs (target E* = 0) ===
  410  <-->  413
  426  <-->  431
  377  <-->  410
  400  <-->  476
  377  <-->  419

=== Contradiction Pairs (target E* = 2) ===
  311  <-->  410
  60  <-->  181
  306  <-->  313
  310  <-->  463
  55  <-->  84


### F.3.3 Sampling a Tension Batch

The `sample_tension_batch` function draws balanced mini-batches with agreement, contradiction and irrelevance examples.

In [8]:
ent_a_ids, ent_b_ids, label_ids = sample_tension_batch(
    training_data.agree_pairs,
    training_data.contra_pairs,
    training_data.n_entities,
    batch_size=12,
)

label_names = {0: 'agreement', 1: 'irrelevance', 2: 'contradiction'}
target_map = {0: 0.0, 1: np.sqrt(2), 2: 2.0}

idx_to_ent = {v: k for k, v in adapter.entity_to_idx.items()}
print(f"Batch size: {len(ent_a_ids)}")
for i in range(min(6, len(ent_a_ids))):
    a_name = idx_to_ent.get(ent_a_ids[i].item(), f"entity_{ent_a_ids[i].item()}")
    b_name = idx_to_ent.get(ent_b_ids[i].item(), f"entity_{ent_b_ids[i].item()}")
    lbl = label_ids[i].item()
    print(f"  {a_name:30s} | {b_name:30s} | {label_names[lbl]} (E*={target_map[lbl]:.3f})")

Batch size: 12
  MDL-009                        | Robustness                     | agreement (E*=0.000)
  MDL-002                        | MDL-005                        | agreement (E*=0.000)
  Low-Risk Models                | Models Under Review            | agreement (E*=0.000)
  MDL-003                        | MDL-005                        | agreement (E*=0.000)
  executive_summary              | model_inventory_summary        | contradiction (E*=2.000)
  data_quality_assessment        | model_inventory_summary        | contradiction (E*=2.000)


---
## F.4 Loss Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| $\lambda_{\text{geo}}$ | 5.0 | Dominant loss for $v$-space |
| $\lambda_{\text{tension}}$ | 2.0 | Comparable gradient magnitude |
| $\lambda_{\text{neg}}$ | 2.0 | $2\times$ penalty on negatives |
| $\gamma$ | 1.5 | Near random baseline on $S^{127}$ |
| $\tau$ | 2.0 | Moderate self-adversarial focus |
| $\lambda_{\text{path}}$ | 0.0 | Enable after $v$-space converges |

In [ ]:
from importlib.resources import files
# Verify config alignment: LossConfig defaults == configs/default.yaml
import yaml
from knowlytix.core.config import LossConfig
from knowlytix.knowledge.config import DocGMSConfig

default_cfg = LossConfig()

with open(str(files("knowlytix.harness").joinpath(str(files("knowlytix.harness").joinpath("configs/default.yaml"))))) as f:
    yaml_cfg = yaml.safe_load(f).get('loss', {})

doc_cfg = DocGMSConfig().loss

fields_to_check = ['lambda_geo', 'lambda_tension', 'lambda_neg', 'gamma', 'adv_temperature']
print(f"{'Field':25s} {'LossConfig':>12s} {'YAML':>12s} {'DocGMSConfig':>12s} {'Match':>8s}")
print("-" * 75)
for f in fields_to_check:
    lc = getattr(default_cfg, f)
    yc = yaml_cfg.get(f, 'N/A')
    dc = getattr(doc_cfg, f)
    match = "YES" if lc == yc == dc else "NO"
    print(f"  {f:23s} {lc:12.1f} {yc:>12} {dc:12.1f} {match:>8s}")

---
## F.5 Numerical Stability

Two numerical pitfalls arise during training:

1. **Tension energy gradient explosion:** $E = \sqrt{2 - 2\langle u_a, u_b \rangle}$ has gradient $\to \infty$ when $\langle u_a, u_b \rangle \to 1$. Fix: clamp argument to $\max(\cdot, 10^{-8})$.
2. **Geodesic distance at poles:** $\arccos(\langle p, q \rangle)$ is non-differentiable at $\pm 1$. Fix: clamp dot product to $[-1+\epsilon, 1-\epsilon]$.

Let's verify the epsilon-safe implementation:

In [10]:
# Demonstrate the epsilon-safe tension energy
def tension_energy_safe(u_a, u_b, eps=1e-8):
    """Epsilon-safe tension energy."""
    dot = (u_a * u_b).sum(dim=-1)
    return torch.sqrt(torch.clamp(2.0 - 2.0 * dot, min=eps))

def tension_energy_unsafe(u_a, u_b):
    """Unsafe -- gradient explosion when dot -> 1."""
    dot = (u_a * u_b).sum(dim=-1)
    return torch.sqrt(2.0 - 2.0 * dot)

# Nearly identical vectors (agreement case)
u = torch.randn(128, device=DEVICE)
u = u / u.norm()
u_a = u.clone().requires_grad_(True)
u_b = (u + 1e-6 * torch.randn_like(u)).requires_grad_(True)
u_b = (u_b / u_b.norm()).detach().requires_grad_(True)

# Safe version
e_safe = tension_energy_safe(u_a, u_b)
e_safe.backward()
grad_safe = u_a.grad.norm().item()

# Unsafe version
u_a2 = u.clone().requires_grad_(True)
e_unsafe = tension_energy_unsafe(u_a2, u_b.detach())
e_unsafe.backward()
grad_unsafe = u_a2.grad.norm().item()

print(f"Safe:   energy={e_safe.item():.6f}, grad_norm={grad_safe:.2f}")
print(f"Unsafe: energy={e_unsafe.item():.6f}, grad_norm={grad_unsafe:.2f}")
print(f"\nGradient ratio (unsafe/safe): {grad_unsafe/max(grad_safe, 1e-10):.1f}x")

Safe:   energy=0.000100, grad_norm=0.00
Unsafe: energy=nan, grad_norm=nan

Gradient ratio (unsafe/safe): nanx


---
## F.6 Post-Training Diagnostics

The nine-check diagnostic suite verifies that the store is production-ready.

### Structural Invariants (S1--S3)

| Check | Verifies | Pass Criterion |
|-------|----------|----------------|
| S1. Cayley orthogonality | $O_r^\top O_r = I$, $\det(O_r) = 1$ | max error $< 10^{-3}$ |
| S2. Stiefel projection | $PP^\top = I$ | $\|PP^\top - I\| < 10^{-2}$ |
| S3. Sphere embedding | $\|v\| = 1$, $\|u\| = 1$ | max deviation $< 10^{-4}$ |

### GMS Primitive Checks (P1--P6)

| Check | Verifies | Pass Criterion |
|-------|----------|----------------|
| P1. `score_triple` | True < fabricated | $\geq 90\%$ |
| P2. `tension_energy` | Agreement/contradiction separation | agree $\geq 80\%$, contra $\geq 70\%$ |
| P3. `check_holonomy` | Triangle defects $\ll$ random | defect $< \sqrt{2m}/2$ |
| P4. `lookup_enm` | SHA-256 integrity | $100\%$ |
| P5. `link_predict` | Filtered rank quality | MRR $\geq 0.3$, H@5 $\geq 50\%$ |
| P6. `query_triples` | Structural correctness | $100\%$ |

In [ ]:
from knowlytix.core.diagnostics import GMSDiagnostics

diag = GMSDiagnostics(
    model=store.model,
    adapter=store.adapter,
    enm=store.enm,
    doc_graph=store.doc_graph,
    device=DEVICE,
)

report = diag.run_all()
GMSDiagnostics.print_report(report)

### F.6.1 Programmatic Access to Results

Each check returns a `CheckResult` with name, passed, score, detail and evidence fields.

In [12]:
print(f"Overall: {'ALL PASSED' if report.all_passed else 'SOME FAILED'}")
print(f"Passed: {report.pass_count}/{report.total}")
print()

for check in report.checks:
    status = "PASS" if check.passed else "FAIL"
    print(f"  [{status}] {check.name:30s} score={check.score:.4f}  {check.detail}")

Overall: SOME FAILED
Passed: 8/9

  [PASS] cayley_orthogonality           score=1.0000  max ||R^T R - I||_F = 2.37e-05, max |det(R) - 1| = 1.34e-04 across 66 rotors
  [PASS] stiefel_projection             score=1.0000  P_v: ||PP^T - I|| = 4.25e-06 (OK), P_u: ||PP^T - I|| = 4.11e-06 (OK)
  [PASS] sphere_embedding               score=1.0000  max v-sphere deviation: 1.19e-07, max u-sphere deviation: 1.19e-07 across 508 entities
  [PASS] score_triple                   score=0.9086  179/197 (90.9%) true triples score lower than type-matched fabricated triples
  [FAIL] tension_energy                 score=0.7150  Agreement < sqrt2: 196/200 (98.0%), Contradiction > sqrt2: 90/200 (45.0%), Spectrum width: 1.565
  [PASS] check_holonomy                 score=0.8418  Identity check: mean defect 0.000022 (OK). Triangle holonomy: 41 triangles, mean defect 2.5307 vs random baseline 16.0 (OK)
  [PASS] lookup_enm                     score=1.0000  261/261 ENM entries pass SHA-256 integrity. 261/261 entr

### F.6.2 Individual Structural Checks

Let's run and inspect each structural check individually.

In [13]:
# S1: Cayley orthogonality
s1 = diag.check_cayley_orthogonality()
print(f"S1 Cayley Orthogonality: {'PASS' if s1.passed else 'FAIL'}")
print(f"   Max OrthoError: {s1.score:.6f} (threshold: 1e-3)")
print(f"   {s1.detail}")

# S2: Stiefel projection
s2 = diag.check_stiefel_projection()
print(f"\nS2 Stiefel Projection: {'PASS' if s2.passed else 'FAIL'}")
print(f"   Error: {s2.score:.6f} (threshold: 1e-2)")
print(f"   {s2.detail}")

# S3: Sphere embedding
s3 = diag.check_sphere_embedding()
print(f"\nS3 Sphere Embedding: {'PASS' if s3.passed else 'FAIL'}")
print(f"   Max deviation: {s3.score:.6f} (threshold: 1e-4)")
print(f"   {s3.detail}")

S1 Cayley Orthogonality: PASS
   Max OrthoError: 1.000000 (threshold: 1e-3)
   max ||R^T R - I||_F = 2.37e-05, max |det(R) - 1| = 1.34e-04 across 66 rotors

S2 Stiefel Projection: PASS
   Error: 1.000000 (threshold: 1e-2)
   P_v: ||PP^T - I|| = 4.25e-06 (OK), P_u: ||PP^T - I|| = 4.11e-06 (OK)

S3 Sphere Embedding: PASS
   Max deviation: 1.000000 (threshold: 1e-4)
   max v-sphere deviation: 1.19e-07, max u-sphere deviation: 1.19e-07 across 508 entities


### F.6.3 Individual Primitive Checks

In [14]:
# P1: score_triple discrimination
p1 = diag.check_score_triple()
print(f"P1 score_triple: {'PASS' if p1.passed else 'FAIL'}")
print(f"   Discrimination: {p1.score:.1%} (threshold: 90%)")
print(f"   {p1.detail}")

# P2: tension_energy
p2 = diag.check_tension_energy()
print(f"\nP2 tension_energy: {'PASS' if p2.passed else 'FAIL'}")
print(f"   Score: {p2.score:.4f}")
print(f"   {p2.detail}")

P1 score_triple: PASS
   Discrimination: 91.8% (threshold: 90%)
   180/196 (91.8%) true triples score lower than type-matched fabricated triples

P2 tension_energy: FAIL
   Score: 0.7150
   Agreement < sqrt2: 196/200 (98.0%), Contradiction > sqrt2: 90/200 (45.0%), Spectrum width: 1.565


In [15]:
# P3: holonomy
p3 = diag.check_holonomy()
print(f"P3 check_holonomy: {'PASS' if p3.passed else 'FAIL'}")
print(f"   Defect: {p3.score:.4f}")
print(f"   {p3.detail}")

# P4: ENM integrity
p4 = diag.check_enm_integrity()
print(f"\nP4 lookup_enm: {'PASS' if p4.passed else 'FAIL'}")
print(f"   Integrity: {p4.score:.1%}")
print(f"   {p4.detail}")

# P5: link prediction
p5 = diag.check_link_prediction()
print(f"\nP5 link_predict: {'PASS' if p5.passed else 'FAIL'}")
print(f"   MRR: {p5.score:.4f}")
print(f"   {p5.detail}")

# P6: query_triples
p6 = diag.check_query_triples()
print(f"\nP6 query_triples: {'PASS' if p6.passed else 'FAIL'}")
print(f"   Correctness: {p6.score:.1%}")
print(f"   {p6.detail}")

P3 check_holonomy: PASS
   Defect: 0.8418
   Identity check: mean defect 0.000022 (OK). Triangle holonomy: 41 triangles, mean defect 2.5307 vs random baseline 16.0 (OK)

P4 lookup_enm: PASS
   Integrity: 100.0%
   261/261 ENM entries pass SHA-256 integrity. 261/261 entries readable.



P5 link_predict: PASS
   MRR: 0.7591
   MRR: 0.7591, Hits@1: 62.0%, Hits@3: 87.5%, Hits@5: 96.0%, Hits@10: 99.5%, Mean rank: 1.9 (n=200)

P6 query_triples: PASS
   Correctness: 100.0%
   400/400 structural queries correct (100.0%)


---
## F.7 Troubleshooting

| Symptom | Root Cause | Remedy |
|---------|-----------|--------|
| P1 fails | Negatives too easy | Enable type-aware hard negatives |
| P2 width < 0.5 | $\lambda_{\text{tension}} = 0$ | Set $\lambda_{\text{tension}} \geq 2.0$ |
| NaN in tension | Gradient explosion | Verify $\epsilon$-clamp in `tension_energy` |
| P5 MRR low | Identity rotor bias | Include self-negative; mask self in link prediction |

### Lab: Demonstrating the Effect of $\lambda_{\text{tension}} = 0$

To see why tension supervision is essential, let's check what tension energies look like on our trained store versus what they would look like without training.

In [16]:
# Compare trained tension energies against random baseline
model = store.model
model.eval()

# Compute tension energies on agreement pairs
agree_energies = []
for a, b in training_data.agree_pairs[:50]:
    if a in adapter.entity_to_idx and b in adapter.entity_to_idx:
        e = store.tension_energy(a, b)
        agree_energies.append(e)

# Compute tension energies on contradiction pairs
contra_energies = []
for a, b in training_data.contra_pairs[:50]:
    if a in adapter.entity_to_idx and b in adapter.entity_to_idx:
        e = store.tension_energy(a, b)
        contra_energies.append(e)

# Random baseline: random unit vectors on S^{m-1} have E ~ sqrt(2)
sqrt2 = np.sqrt(2)

print(f"Agreement energies:     mean={np.mean(agree_energies):.3f}  (target: 0.0)")
print(f"Contradiction energies: mean={np.mean(contra_energies):.3f}  (target: 2.0)")
print(f"Random baseline:        {sqrt2:.3f}")
print(f"\nSeparation width: {np.mean(contra_energies) - np.mean(agree_energies):.3f}")
print(f"Agree < sqrt(2): {np.mean([e < sqrt2 for e in agree_energies]):.1%}")
print(f"Contra > sqrt(2): {np.mean([e > sqrt2 for e in contra_energies]):.1%}")

Agreement energies:     mean=nan  (target: 0.0)
Contradiction energies: mean=nan  (target: 2.0)
Random baseline:        1.414

Separation width: nan
Agree < sqrt(2): nan%
Contra > sqrt(2): nan%


---
## Summary

In this appendix notebook we:

1. **Prepared training data** -- positive triples, type-aware hard negatives, tension pairs and relation triangles.
2. **Verified config alignment** -- `LossConfig`, `DocGMSConfig` and `configs/default.yaml` share identical validated defaults.
3. **Demonstrated numerical stability** -- epsilon-safe tension energy prevents gradient explosion.
4. **Ran the 9-check diagnostic suite** -- 3 structural invariants + 6 primitive checks.
5. **Inspected individual diagnostics** -- programmatic access to each check result.

These diagnostics should be run after every training run before the store enters production.